# Get 1000 ADS-B registrations across Europe


In [72]:
from datetime import date
import math
import random

import geopandas as gpd
import pandas as pd
import polars as pl

from data_engineering.adsb.read_adsb import read_adsb

TARGET_DATE = date(2026, 6, 1)
N_PLANES = 1000
RANDOM_SEED = 42
COUNTRIES_PATH = "/Volumes/T2-SSD/planequery/data/raw/natural-earth/ne_10m_admin_0_countries/ne_10m_admin_0_countries.shp"

EUROPE_ISO3 = {
    "ALB", "AND", "AUT", "BEL", "BIH", "BGR", "BLR", "CHE", "CYP",
    "CZE", "DEU", "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC",
    "HRV", "HUN", "IRL", "ISL", "ITA", "LIE", "LTU", "LUX", "LVA",
    "MCO", "MDA", "MKD", "MLT", "MNE", "NLD", "NOR", "POL", "PRT",
    "ROU", "SMR", "SRB", "SVK", "SVN", "SWE", "UKR", "VAT",
}


In [73]:
df = read_adsb(TARGET_DATE)

countries = gpd.read_file(COUNTRIES_PATH).to_crs("EPSG:4326")
europe = (
    countries[countries["ADM0_A3"].isin(EUROPE_ISO3)]
    [["ADM0_A3", "ADMIN", "geometry"]]
    .rename(columns={"ADM0_A3": "ISO_A3"})
)


In [74]:
minx, miny, maxx, maxy = europe.total_bounds

adsb_points = (
    df
    .select(["registration", "lat", "lon"])
    .with_columns(
        pl.col("registration").cast(pl.String).str.strip_chars(),
        pl.col("lat").cast(pl.Float64, strict=False),
        pl.col("lon").cast(pl.Float64, strict=False),
    )
    .drop_nulls(["registration", "lat", "lon"])
    .filter(pl.col("registration") != "")
    .filter(
        pl.col("lat").is_between(miny, maxy)
        & pl.col("lon").is_between(minx, maxx)
    )
    .unique()
    .to_pandas()
)

points = gpd.GeoDataFrame(
    adsb_points,
    geometry=gpd.points_from_xy(adsb_points["lon"], adsb_points["lat"]),
    crs="EPSG:4326",
)

joined = gpd.sjoin(points, europe, how="inner", predicate="intersects")

registration_country = (
    joined[["registration", "ISO_A3"]]
    .dropna()
    .drop_duplicates()
)

planes_per_country = (
    registration_country
    .groupby("ISO_A3")["registration"]
    .nunique()
    .reindex(sorted(EUROPE_ISO3), fill_value=0)
    .reset_index(name="num_planes")
    .sort_values("num_planes", ascending=False)
)

planes_per_country


,ISO_A3,num_planes
10,DEU,5443
15,FRA,4817
16,GBR,3669
22,ITA,3369
12,ESP,2755
2,AUT,2700
7,CHE,2356
32,NLD,2154
3,BEL,2107
19,HUN,1872


In [75]:
registration_to_countries = (
    registration_country
    .groupby("registration")["ISO_A3"]
    .agg(set)
    .to_dict()
)

country_to_registrations = (
    registration_country
    .groupby("ISO_A3")["registration"]
    .agg(set)
    .to_dict()
)

missing_countries = sorted(EUROPE_ISO3 - set(country_to_registrations))
if missing_countries:
    raise ValueError(f"Cannot cover countries with no planes: {missing_countries}")

if len(registration_to_countries) < N_PLANES:
    raise ValueError(f"Only {len(registration_to_countries)} unique registrations available")


In [80]:
def proportional_country_quota(registration_country: pd.DataFrame, n: int) -> dict[str, int]:
    counts = registration_country.groupby("ISO_A3")["registration"].nunique()
    raw = counts / counts.sum() * n
    quota = raw.apply(math.floor).clip(lower=1)

    remaining = n - int(quota.sum())
    for country in (raw - raw.apply(math.floor)).sort_values(ascending=False).index[:remaining]:
        quota[country] += 1

    return quota.astype(int).to_dict()


def select_registrations(n: int = N_PLANES, seed: int = RANDOM_SEED) -> list[str]:
    rng = random.Random(seed)
    quota = proportional_country_quota(registration_country, n)
    selected: set[str] = set()

    # Guarantee country coverage first, beginning with countries that have fewer options.
    for country, regs in sorted(country_to_registrations.items(), key=lambda item: len(item[1])):
        if selected.intersection(regs):
            continue

        selected.add(max(
            regs,
            key=lambda reg: sum(
                1 / len(country_to_registrations[country])
                for country in registration_to_countries[reg]
            ),
        ))

    while len(selected) < n:
        selected_country_counts = (
            registration_country[registration_country["registration"].isin(selected)]
            .groupby("ISO_A3")["registration"]
            .nunique()
            .to_dict()
        )
        deficits = {
            country: quota[country] - selected_country_counts.get(country, 0)
            for country in quota
        }
        candidates = list(set(registration_to_countries) - selected)

        def score(registration: str) -> int:
            return sum(
                max(deficits.get(country, 0), 0)
                for country in registration_to_countries[registration]
            )

        best_score = max(score(registration) for registration in candidates)
        if best_score <= 0:
            selected.add(rng.choice(candidates))
            continue

        selected.add(rng.choice([
            registration for registration in candidates
            if score(registration) == best_score
        ]))

    return sorted(selected)


selected_registrations = select_registrations()
selected_planes = pd.DataFrame({"registration": selected_registrations})
selected_planes


,registration
0,03
1,07-8614
2,09-20187
3,10-20309
4,14-08155
...,...
995,ZM322
996,ZZ337
997,ZZ394
998,ZZ396


In [77]:
selected_planes_per_country = (
    registration_country[registration_country["registration"].isin(selected_registrations)]
    .groupby("ISO_A3")["registration"]
    .nunique()
    .reindex(sorted(EUROPE_ISO3), fill_value=0)
    .reset_index(name="num_selected_planes")
    .sort_values("num_selected_planes", ascending=False)
)

print("selected registrations:", len(selected_registrations))
print("covered countries:", (selected_planes_per_country["num_selected_planes"] > 0).sum())

selected_planes_per_country


selected registrations: 1000
covered countries: 44


,ISO_A3,num_selected_planes
10,DEU,513
15,FRA,414
22,ITA,333
16,GBR,329
2,AUT,290
12,ESP,250
7,CHE,236
3,BEL,215
32,NLD,208
19,HUN,189


In [79]:
selected_planes

,registration
0,03
1,07-8614
2,09-20187
3,10-20309
4,14-08155
...,...
995,ZM322
996,ZZ337
997,ZZ394
998,ZZ396
